Decorators are one of Python's most powerful, elegant, and advanced features. They allow you to modify or enhance the behavior of functions or classes without permanently changing their source code. They are widely used for logging, access control, caching, and execution timing.

## 1. Function Decorators
At its core, a decorator is a higher-order function: a function that takes another function as an argument, adds some functionality, and returns a new function.

To use a decorator cleanly, Python provides syntactic sugar using the @ symbol placed directly above the target function definition.

In [ ]:
# A simple decorator function
def my_decorator(func):
  def wrapper():
    print("-> Something is happening BEFORE the function is called.")
    func()  # Execute the original function
    print("-> Something is happening AFTER the function is called.")

  return wrapper


# Applying the decorator using syntactic sugar
@my_decorator
def say_hello():
  print("Hello!")


# Calling the decorated function
say_hello()
# Output:
# -> Something is happening BEFORE the function is called.
# Hello!
# -> Something is happening AFTER the function is called.

# 2. Preserving Metadata with functools.wraps
When you wrap a function inside a decorator, the original function's metadata (such as its \_\_name\_\_ and \_\_doc\_\_ docstring) gets overwritten by the inner wrapper function.

To fix this and preserve the original function's identity, you must use functools.wraps.

In [ ]:
from functools import wraps


def explicit_decorator(func):
  @wraps(func)  # Preserves metadata
  def wrapper(*args, **kwargs):
    """This is the wrapper docstring"""
    return func(*args, **kwargs)

  return wrapper


@explicit_decorator
def compute_sum(a, b):
  """Adds two numbers together."""
  return a + b


print(compute_sum.__name__)  # Output: compute_sum (without wraps, it would be 'wrapper')
print(compute_sum.__doc__)  # Output: Adds two numbers together.

# 3. Decorators with Arguments
Sometimes you want to pass parameters into the decorator itself (e.g., @repeat(num_times=3)). This requires adding an extra layer of nesting—turning your decorator into a "decorator factory" that accepts arguments and returns the actual decorator.

In [ ]:
from functools import wraps


# 1. Outer function accepts decorator arguments
def repeat(num_times):
  def decorator_repeat(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
      for _ in range(num_times):
        result = func(*args, **kwargs)
      return result

    return wrapper

  return decorator_repeat


# 2. Using the parameterized decorator
@repeat(num_times=3)
def greet(name):
  print(f"Hello, {name}!")


greet("Alice")
# Output:
# Hello, Alice!
# Hello, Alice!
# Hello, Alice!

# 4. Nested (Stacked) Decorators
You can apply multiple decorators to a single function by stacking them vertically.

### Execution Order
Stacking order matters! Decorators are executed from the bottom up (closest to the function first), but applied from the top down.

In [ ]:
from functools import wraps


def bold(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
    return f"<b>{func(*args, **kwargs)}</b>"

  return wrapper


def italic(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
    return f"<i>{func(*args, **kwargs)}</i>"

  return wrapper


# Stacking decorators
@bold
@italic
def get_text(name):
  return f"Hello {name}"


print(get_text("Bob"))
# Output: <b><i>Hello Bob</i></b>
# (italic runs first, wrapping in <i>, then bold runs second, wrapping in <b>)

# 5. Class Decorators
Decorators are not limited to functions; they can also be implemented as classes. To turn a class into a decorator, you use the \_\_call\_\_ dunder method so that instances of the class can behave like functions.

In [ ]:
from functools import wraps


class CallCounter:

  def __init__(self, func):
    self.func = func
    self.num_calls = 0
    wraps(func)(self)  # Preserves metadata

  def __call__(self, *args, **kwargs):
    self.num_calls += 1
    print(f"Function '{self.func.__name__}' has been called {self.num_calls} times.")
    return self.func(*args, **kwargs)


@CallCounter
def say_hi():
  print("Hi!")


say_hi()  # Output: Function 'say_hi' has been called 1 times. \n Hi!
say_hi()  # Output: Function 'say_hi' has been called 2 times. \n Hi!